# APAN Final Project — AI-Assisted Security Analysis

**Team Name:** _Group 1_  
**Students:** _Aditi Iyer (ai2584), Harn Hsiao (hh3169), Helen Liu (hl3909), Yinuo Xu (yx3028)_  
**Date:** _2026-05-04_  
**Repository Analyzed:** `final_project/securenote`

This notebook documents the end-to-end workflow used to analyze the SecureNote codebase, validate the scanner output, consolidate findings, and export the final CSV in the required schema: `file, lines, severity, evidence, confidence, mitigation`.

## 2.1 Scanner Implementation

The scanner reads `.py` files, checks each line against a set of security-focused patterns, adds targeted checks for path traversal and missing admin authentication, and stores raw findings in a DataFrame.

In [ ]:
import os
import re
import pandas as pd
import matplotlib.pyplot as plt

def print_tree(start_path="."):
    for root, dirs, files in os.walk(start_path):
        if "venv" in root:
            continue
        level = root.replace(start_path, "").count(os.sep)
        indent = " " * 4 * level
        print(f"{indent}{os.path.basename(root)}/")
        subindent = " " * (4 * (level + 1))
        for f in files:
            print(f"{subindent}{f}")

print_tree()

./
    .gitignore
    .Rhistory
    app.py
    config.py
    findings.csv
    init_db.py
    README.md
    requirements.txt
    securenote_security_analysis.ipynb
    securenote_security_analysis_HLcheck.ipynb
    securenote_security_analysis_validation.ipynb
    securenote_security_analysis_validation_HLcheck.ipynb
    test_output.txt
    test_routes.py
    .git/
        COMMIT_EDITMSG
        config
        description
        FETCH_HEAD
        HEAD
        index
        ORIG_HEAD
        hooks/
            applypatch-msg.sample
            commit-msg.sample
            fsmonitor-watchman.sample
            post-update.sample
            pre-applypatch.sample
            pre-commit.sample
            pre-merge-commit.sample
            pre-push.sample
            pre-rebase.sample
            pre-receive.sample
            prepare-commit-msg.sample
            push-to-checkout.sample
            sendemail-validate.sample
            update.sample
        info/
            exclude
  

In [2]:
py_files = []
total_lines = 0

for root, dirs, files in os.walk("."):
    if "venv" in root:
        continue
    for file in files:
        if file.endswith(".py"):
            path = os.path.join(root, file)
            py_files.append(path)
            with open(path, "r", encoding="utf-8", errors="ignore") as f:
                total_lines += sum(1 for _ in f)

print("Python files:", len(py_files))
print("Total lines:", total_lines)

for f in py_files:
    print(f)

Python files: 14
Total lines: 955
.\app.py
.\config.py
.\init_db.py
.\test_routes.py
.\models\note.py
.\models\user.py
.\models\__init__.py
.\routes\admin.py
.\routes\auth.py
.\routes\export.py
.\routes\notes.py
.\routes\__init__.py
.\utils\files.py
.\utils\__init__.py


## 1. Scope and Repository Overview

SecureNote is a Flask-based REST API. This section summarizes the repository structure, app purpose, key components, and highest-risk areas selected for deeper analysis.

### Key Components

- app.py: main application entry point
- routes/: handles API endpoints (auth, notes, admin, export)
- models/: database structure and data handling
- utils/: helper functions such as file handling
- templates/: HTML views
- uploads/: stored uploaded files

### Attack Surface

The main attack surface includes endpoints that accept user input. This includes authentication routes, note creation and search endpoints, export functionality, and file download routes.

Key exposed endpoints include:
- /register, /login
- /notes
- /notes/search
- /export
- /files/<filename>
- /admin/...

These areas are most likely to contain vulnerabilities such as injection attacks, improper input validation, and insecure file handling.

### High-Risk File: `utils/files.py`

In [3]:
with open("utils/files.py", "r") as f:
    print(f.read())

"""
SecureNote â€“ File Utilities
Helper functions for serving uploaded or generated files.
"""

import os
from flask import send_file, abort

from config import Config


def download_file(filename: str):
    """Serve a file from the uploads directory.

    Parameters
    ----------
    filename : str
        Name (or relative path) of the requested file.
    """
    filepath = os.path.join(Config.UPLOAD_FOLDER, filename)

    if not os.path.isfile(filepath):
        abort(404, description="File not found")

    return send_file(filepath, as_attachment=True)



- `utils/files.py`: Handles file download functionality
- This is a high-risk area because it constructs file paths using user-controlled input without validation
- This may introduce a path traversal vulnerability, allowing attackers to access files outside the intended directory

### High-Risk File: `routes/auth.py`

In [4]:
with open("routes/auth.py", "r") as f:
    print(f.read())

"""
SecureNote â€“ Authentication Routes
Handles user registration, login, and logout.
"""

import logging
from functools import wraps

from flask import (
    Blueprint,
    request,
    jsonify,
    session,
    redirect,
    url_for,
)

from models.user import create_user, get_user_by_username, verify_password

logger = logging.getLogger(__name__)

auth_bp = Blueprint("auth", __name__)


# â”€â”€ Helpers â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def login_required(f):
    """Decorator that rejects unauthenticated requests."""
    @wraps(f)
    def wrapper(*args, **kwargs):
        if "user_id" not in session:
            return jsonify({"error": "Authentication required"}), 401
        return f(*args, **kwargs)
    return wrapper


def admin_required(f):
    """Decorator that restricts access to admin users."""
    @wraps(f)
    @

- `routes/auth.py`: Handles user registration, login, and session management
- This is a high-risk area because it processes authentication data
- The login route logs raw passwords during login attempts, which may expose sensitive credentials if logs are accessed or leaked

### High-Risk File: `routes/notes.py`

In [5]:
with open("routes/notes.py", "r") as f:
    print(f.read())

"""
SecureNote â€“ Notes Routes
CRUD endpoints and search for user notes.
"""

import logging

from flask import Blueprint, request, jsonify, session, render_template

from models.note import (
    create_note,
    get_note,
    get_notes_for_user,
    update_note,
    delete_note,
    search_notes,
)
from routes.auth import login_required

logger = logging.getLogger(__name__)

notes_bp = Blueprint("notes", __name__)


# â”€â”€ List & Search â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

@notes_bp.route("/notes", methods=["GET"])
@login_required
def list_notes():
    """Return all notes for the logged-in user."""
    user_id = session["user_id"]
    notes = get_notes_for_user(user_id)
    return jsonify([dict(n) for n in notes])


@notes_bp.route("/notes/search", methods=["GET"])
@login_required
def search():
    """Full-text search across note titles for the

- `routes/notes.py`: Handles note creation, search, viewing, updating, and deletion
- This is a high-risk area because it processes user-generated content and renders note data into HTML
- Rendering stored note content in a template may introduce cross-site scripting (XSS) risk if output is not properly escaped

### High-Risk File: `routes/export.py`

In [6]:
with open("routes/export.py", "r") as f:
    print(f.read())

"""
SecureNote â€“ Export / Import Routes
Endpoints for exporting notes to various formats and importing note
collections from serialized payloads.
"""

import base64
import logging
import os
import pickle
import tempfile

from flask import Blueprint, request, jsonify, session, send_file

from models.note import get_notes_for_user, get_note
from routes.auth import login_required

logger = logging.getLogger(__name__)

export_bp = Blueprint("export", __name__, url_prefix="/export")


@export_bp.route("/csv", methods=["GET"])
@login_required
def export_csv():
    """Export all of the current user's notes as a CSV file."""
    import csv
    import io

    notes = get_notes_for_user(session["user_id"])

    buf = io.StringIO()
    writer = csv.writer(buf)
    writer.writerow(["id", "title", "content", "created_at"])
    for n in notes:
        writer.writerow([n["id"], n["title"], n["content"], n["created_at"]])

    output = io.BytesIO(buf.getvalue().encode("utf-8"))
    output.seek(0)
  

- `routes/export.py`: Handles note export and import functionality
- This is a high-risk area because it processes serialized data, builds shell commands, and inserts user-controlled content into generated files
- The import route uses unsafe pickle deserialization, and the PDF export route may introduce command injection risk through unsanitized filename input

### High-Risk File: `routes/admin.py`

In [7]:
with open("routes/admin.py", "r") as f:
    print(f.read())

"""
SecureNote â€“ Admin Routes
Endpoints for user management (listing, deleting).
These are intended for administrator use only.
"""

import logging

from flask import Blueprint, jsonify, request

from models.user import list_all_users, delete_user, get_user_by_id

logger = logging.getLogger(__name__)

admin_bp = Blueprint("admin", __name__, url_prefix="/admin")


@admin_bp.route("/users", methods=["GET"])
def get_all_users():
    """List every registered user (id, username, role)."""
    users = list_all_users()
    return jsonify([dict(u) for u in users])


@admin_bp.route("/users/<int:user_id>", methods=["GET"])
def get_user(user_id):
    """Retrieve a single user's details."""
    user = get_user_by_id(user_id)
    if user is None:
        return jsonify({"error": "User not found"}), 404
    return jsonify({
        "id": user["id"],
        "username": user["username"],
        "role": user["role"],
    })


@admin_bp.route("/users/<int:user_id>", methods=["DELETE"])
def remove_u

- `routes/admin.py`: Handles administrative user management functions such as listing users, deleting users, and changing roles
- This is a high-risk area because it exposes sensitive administrative actions
- The admin routes do not enforce authentication or admin authorization, which may allow unauthorized users to access or modify user accounts

### High-Risk File: `models/user.py`

In [8]:
with open("models/user.py", "r") as f:
    print(f.read())

"""
SecureNote â€“ User Model
Handles user creation, password storage, and authentication lookups.
"""

import hashlib
import sqlite3

from config import Config


def get_db():
    """Return a connection to the SQLite database."""
    conn = sqlite3.connect(Config.DATABASE_PATH)
    conn.row_factory = sqlite3.Row
    return conn


# â”€â”€ Password helpers â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€

def hash_password(password: str) -> str:
    """Return a hex-digest hash of *password* for storage."""
    return hashlib.md5(password.encode()).hexdigest()


def verify_password(password: str, stored_hash: str) -> bool:
    """Check a plain-text *password* against a *stored_hash*."""
    return hash_password(password) == stored_hash


# â”€â”€ CRUD â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”

- `models/user.py`: Handles user creation, password storage, and authentication lookups
- This is a high-risk area because it manages credential security
- Passwords are hashed using MD5, which is an outdated and insecure hashing algorithm that may make stored credentials easier to crack if the database is compromised

### High-Risk File: `config.py`

In [9]:
with open("config.py", "r") as f:
    print(f.read())

"""
SecureNote â€“ Application Configuration
Centralized configuration for the SecureNote Flask application.
"""

import os


class Config:
    """Base configuration."""

    # â”€â”€ Security â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    SECRET_KEY = "supersecretkey123"                    # used for session signing
    SESSION_COOKIE_HTTPONLY = True

    # â”€â”€ Database â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    DATABASE_PATH = os.path.join(
        os.path.dirname(os.path.abspath(__file__)), "securenote.db"
    )

    # â”€â”€ Upload / Export â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
    UP

- `config.py`: Handles application-wide security, database, upload, and logging settings
- This is a high-risk area because insecure configuration can expose the entire application
- The file contains a hardcoded secret key and enables debug mode in the production configuration, which may weaken session security and expose sensitive internal details

### Priority Files for Analysis

Based on the repository review, the following files are the highest priority for vulnerability analysis:

1. `routes/export.py` – unsafe deserialization and command execution
2. `routes/admin.py` – missing authentication and authorization
3. `utils/files.py` – potential path traversal
4. `models/user.py` – weak password hashing
5. `routes/auth.py` – credential exposure in logs

These files represent the most critical areas of the application where high-severity vulnerabilities are likely to exist.

## 2. Methodology: Rule-Based Static Analysis

The first analysis layer uses lightweight regular expressions and file-specific logic to identify known risky patterns in the SecureNote source code.

### Goal

Build a lightweight rule-based static analysis script to automatically scan the SecureNote codebase for insecure coding patterns identified during Phase 1.

The objective is to generate a raw findings table for manual validation, not to build a complete or fully accurate vulnerability scanner.

### Rule-Based Approach

The scanner performs a recursive search across all Python files in the repository and applies simple pattern-matching rules using regular expressions.

Each match is recorded with:
- file path
- line number
- vulnerability type
- code snippet (evidence)

This approach prioritizes simplicity and transparency over detection accuracy.

### Detection Rules

The following rules are based on vulnerabilities identified during Phase 1 manual analysis:

- `pickle.loads` → unsafe deserialization
- `os.system(` → command execution risk
- `md5(` → weak password hashing
- `DEBUG = True` → insecure configuration
- `SECRET_KEY =` → hardcoded secret
- logging of passwords → sensitive data exposure

### Codes

### Imports

In [10]:
import os
import re
import pandas as pd

### Rules

In [11]:
RULES = [
    {
        "vulnerability_type": "Unsafe deserialization",
        "pattern": r"pickle\.loads\s*\(",
    },
    {
        "vulnerability_type": "Command execution",
        "pattern": r"os\.system\s*\(",
    },
    {
        "vulnerability_type": "Weak hashing",
        "pattern": r"(hashlib\.)?md5\s*\(",
    },
    {
        "vulnerability_type": "Debug mode enabled",
        "pattern": r"DEBUG\s*=\s*True",
    },
    {
        "vulnerability_type": "Hardcoded secret",
        "pattern": r"SECRET_KEY\s*=",
    },
    {
        "vulnerability_type": "Password logging",
        "pattern": r"(logger|logging)\.\w+\(.*password.*\)",
    },
]

### Scan One File

In [12]:
def scan_file(file_path):
    findings = []

    try:
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            lines = f.readlines()
    except:
        return findings

    for i, line in enumerate(lines, start=1):
        for rule in RULES:
            if re.search(rule["pattern"], line):
                findings.append({
                    "file": file_path,
                    "line_number": i,
                    "vulnerability_type": rule["vulnerability_type"],
                    "evidence": line.strip()
                })

    return findings

### Detect Possible Path Traversal

In [13]:
def detect_path_traversal(file_path):
    findings = []

    try:
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            lines = f.readlines()
    except:
        return findings

    suspicious_path_terms = ["filename", "path", "request.args", "request.form", "request.files"]
    file_ops = ["open(", "send_file(", "os.path.join(", "os.path.isfile("]

    for i, line in enumerate(lines, start=1):
        if any(op in line for op in file_ops) and any(term in line for term in suspicious_path_terms):
            findings.append({
                "file": file_path,
                "line_number": i,
                "vulnerability_type": "Possible path traversal",
                "evidence": line.strip()
            })

    return findings

### Detect Missing Auth on Admin Routes

In [14]:
def detect_missing_admin_auth(file_path):
    findings = []

    if "admin.py" not in file_path:
        return findings

    try:
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            lines = f.readlines()
    except:
        return findings

    for i, line in enumerate(lines):
        if "@admin_bp.route" in line:
            context = "".join(lines[max(0, i-3):i+3])

            if "@login_required" not in context and "@admin_required" not in context:
                findings.append({
                    "file": file_path,
                    "line_number": i + 1,
                    "vulnerability_type": "Missing auth decorator on admin route",
                    "evidence": line.strip()
                })

    return findings

### Scan Entire Repo

In [15]:
def scan_repo(root_dir="."):
    all_findings = []

    for root, dirs, files in os.walk(root_dir):
        if "venv" in root or "__pycache__" in root:
            continue

        for file in files:
            if file.endswith(".py"):
                file_path = os.path.join(root, file)

                all_findings.extend(scan_file(file_path))
                all_findings.extend(detect_path_traversal(file_path))
                all_findings.extend(detect_missing_admin_auth(file_path))

    return all_findings

## 2.2 Run Scanner and Export Raw Findings

This cell generates the raw scanner output. The final submission CSV is produced later after validation, de-duplication, and schema cleanup.

In [16]:
import pandas as pd
findings = scan_repo(".")

df = pd.DataFrame(findings)

if not df.empty:
    df = df.sort_values(by=["file", "line_number"]).reset_index(drop=True)
    df = df.drop_duplicates()

df

,file,line_number,vulnerability_type,evidence
0,.\config.py,14,Hardcoded secret,"SECRET_KEY = ""supersecretkey123"" ..."
1,.\config.py,18,Possible path traversal,DATABASE_PATH = os.path.join(
2,.\config.py,23,Possible path traversal,UPLOAD_FOLDER = os.path.join(
3,.\config.py,36,Debug mode enabled,DEBUG = True
4,.\config.py,42,Debug mode enabled,DEBUG = True ...
5,.\models\user.py,24,Weak hashing,return hashlib.md5(password.encode()).hexdigest()
6,.\routes\admin.py,19,Missing auth decorator on admin route,"@admin_bp.route(""/users"", methods=[""GET""])"
7,.\routes\admin.py,26,Missing auth decorator on admin route,"@admin_bp.route(""/users/<int:user_id>"", method..."
8,.\routes\admin.py,39,Missing auth decorator on admin route,"@admin_bp.route(""/users/<int:user_id>"", method..."
9,.\routes\admin.py,51,Missing auth decorator on admin route,"@admin_bp.route(""/users/<int:user_id>/role"", m..."


### Save Findings

In [17]:
df.to_csv("findings.csv", index=False)
print("Saved findings to findings.csv")

Saved findings to findings.csv


### Summary

In [18]:
df["vulnerability_type"].value_counts()

vulnerability_type
Possible path traversal                  7
Missing auth decorator on admin route    4
Debug mode enabled                       2
Hardcoded secret                         1
Weak hashing                             1
Command execution                        1
Unsafe deserialization                   1
Name: count, dtype: int64

### Findings

The rule-based scanner automatically identified multiple instances of insecure coding patterns across the SecureNote codebase. These include unsafe deserialization, command execution, weak password hashing, insecure configuration, and missing access control.

The findings are consistent with vulnerabilities identified during Phase 1, including the use of pickle.loads in data import, os.system in PDF generation, MD5-based password hashing, hardcoded secret keys, debug mode enabled in production settings, and missing authentication on administrative routes.

Overall, the scanner successfully captured the major high-risk areas of the application and generated a preliminary set of findings for further validation.

### Limitations

The scanner relies on simple pattern matching using regular expressions and does not perform deep code analysis. As a result, it may produce false positives. For example, the path traversal rule flagged file path construction in configuration files even when not directly influenced by user input.

Additionally, the scanner may miss vulnerabilities that do not match predefined patterns or require contextual understanding, such as complex access control logic or indirect data flow issues.

These limitations are expected given the lightweight design of the scanner. All findings will be manually reviewed and validated in the next phase.

In [19]:
def detect_missing_admin_auth(file_path):
    findings = []

    if "admin.py" not in file_path:
        return findings

    try:
        with open(file_path, "r", encoding="utf-8", errors="ignore") as f:
            lines = f.readlines()
    except:
        return findings

    for i, line in enumerate(lines):
        if "@admin_bp.route" in line:
            context = "".join(lines[max(0, i-3):i+3])

            if "@login_required" not in context and "@admin_required" not in context:
                findings.append({
                    "file": file_path,
                    "line_number": i + 1,
                    "vulnerability_type": "Missing auth decorator on admin route",
                    "evidence": line.strip()
                })

    return findings

## 3: Validation & Analysis

### Goal

Review each finding from Phase 2 and determine whether it represents a real vulnerability.

The objective is to confirm or dismiss each scanner finding, explain why it is a vulnerability, and describe its potential impact.

### Approach

Each finding from findings.csv is grouped by vulnerability type and reviewed manually against the source code.

For each group:
- The relevant code is read directly from the file
- The finding is marked as confirmed or dismissed
- A plain-language explanation and impact assessment is recorded

This phase also identifies two vulnerabilities that the scanner did not detect.

### Reviewing the Scanner Output

Before validating, we reload the findings from Phase 2 to use as our reference.

In [20]:
import pandas as pd

# Reload the Phase 2 scanner output
df = pd.read_csv("findings.csv")

# Show a grouped summary of what the scanner flagged
print("Scanner findings by type:\n")
print(df["vulnerability_type"].value_counts().to_string())
print(f"\nTotal rows: {len(df)}")

Scanner findings by type:

vulnerability_type
Possible path traversal                  7
Missing auth decorator on admin route    4
Debug mode enabled                       2
Hardcoded secret                         1
Weak hashing                             1
Command execution                        1
Unsafe deserialization                   1

Total rows: 17


### Validation

Each finding is reviewed below. The code responsible for each vulnerability is read directly from the source file, and a verdict is recorded.

### Finding 1: Unsafe Deserialization
- **File:** `routes/export.py`
- **Scanner finding:** `pickle.loads` called on user-supplied input
- **Confirmed:** Yes

In [21]:
# Read the relevant section of export.py to confirm the finding

with open("routes/export.py", "r") as f:
    lines = f.readlines()

# Print the import_notes function where pickle.loads appears
print("routes/export.py -- import_notes function:\n")
for i, line in enumerate(lines, start=1):
    if 80 <= i <= 100:
        print(f"{i:>4}  {line}", end="")

routes/export.py -- import_notes function:

  80  
  81      The payload should be a base64-encoded pickle of a list of dicts, each
  82      containing 'title' and 'content' keys.
  83      """
  84      data = request.get_json(force=True)
  85      payload = data.get("payload")
  86  
  87      if not payload:
  88          return jsonify({"error": "Missing payload"}), 400
  89  
  90      try:
  91          raw = base64.b64decode(payload)
  92          notes_data = pickle.loads(raw)
  93      except Exception as exc:
  94          logger.error("Import failed: %s", exc)
  95          return jsonify({"error": "Invalid payload format"}), 400
  96  
  97      from models.note import create_note
  98  
  99      imported = 0
 100      for item in notes_data:


- The `/export/import` endpoint accepts a JSON body containing a `payload` field
- The payload is base64-decoded and passed directly to `pickle.loads()`
- `pickle` can execute arbitrary Python code during deserialization
- An attacker can craft a malicious payload that runs any command on the server
- The base64 encoding is not a security control -- it is just a data format
- **Impact:** Remote Code Execution (RCE) -- an authenticated attacker can run arbitrary commands on the server

### Finding 2: Command Injection
- **File:** `routes/export.py`
- **Scanner finding:** `os.system()` called with user-controlled input
- **Confirmed:** Yes

In [22]:
# Read the relevant section of export.py to confirm the finding

with open("routes/export.py", "r") as f:
    lines = f.readlines()

# Print the export_pdf function where os.system appears
print("routes/export.py -- export_pdf function:\n")
for i, line in enumerate(lines, start=1):
    if 55 <= i <= 75:
        print(f"{i:>4}  {line}", end="")

routes/export.py -- export_pdf function:

  55          return jsonify({"error": "Note not found"}), 404
  56  
  57      # Build a temporary HTML file and convert to PDF
  58      html_content = f"<h1>{note['title']}</h1><p>{note['content']}</p>"
  59      tmp_html = tempfile.mktemp(suffix=".html")
  60      with open(tmp_html, "w") as fh:
  61          fh.write(html_content)
  62  
  63      output_filename = request.args.get("filename", f"note_{note_id}")
  64      output_path = os.path.join(tempfile.gettempdir(), f"{output_filename}.pdf")
  65  
  66      # Convert HTML â†’ PDF
  67      cmd = f"wkhtmltopdf {tmp_html} {output_path}"
  68      os.system(cmd)
  69  
  70      if not os.path.exists(output_path):
  71          return jsonify({"error": "PDF generation failed"}), 500
  72  
  73      return send_file(output_path, mimetype="application/pdf", as_attachment=True)
  74  
  75  


- The `/export/pdf/<id>` endpoint accepts a `filename` query parameter from the URL
- This value is inserted directly into a shell command string with no sanitization
- `os.system()` executes the resulting string in a shell
- A request like `/export/pdf/1?filename=out; id` would run two shell commands
- **Impact:** Remote Code Execution (RCE) -- an authenticated attacker can inject arbitrary shell commands

### Finding 3: Missing Authentication on Admin Routes
- **File:** `routes/admin.py`
- **Scanner finding:** admin routes have no `@login_required` or `@admin_required` decorator
- **Confirmed:** Yes

In [23]:
# Compare a protected route in notes.py against an unprotected admin route

print("routes/notes.py -- example of a correctly protected route:\n")
with open("routes/notes.py", "r") as f:
    for i, line in enumerate(f.readlines(), start=1):
        if 24 <= i <= 30:
            print(f"{i:>4}  {line}", end="")

print("\n\nroutes/admin.py -- admin route with no auth decorator:\n")
with open("routes/admin.py", "r") as f:
    for i, line in enumerate(f.readlines(), start=1):
        if 17 <= i <= 24:
            print(f"{i:>4}  {line}", end="")

routes/notes.py -- example of a correctly protected route:

  24  
  25  
  26  # â”€â”€ List & Search â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
  27  
  28  @notes_bp.route("/notes", methods=["GET"])
  29  @login_required
  30  def list_notes():


routes/admin.py -- admin route with no auth decorator:

  17  
  18  
  19  @admin_bp.route("/users", methods=["GET"])
  20  def get_all_users():
  21      """List every registered user (id, username, role)."""
  22      users = list_all_users()
  23      return jsonify([dict(u) for u in users])
  24  


- The `@login_required` and `@admin_required` decorators are defined in `routes/auth.py` and used correctly in other parts of the app
- None of the four routes in `routes/admin.py` apply either decorator
- Any unauthenticated request to `/admin/users` returns the full user list
- Any unauthenticated request to `/admin/users/<id>/role` can promote any account to admin
- **Impact:** Broken access control -- any user, including unauthenticated visitors, can list users, delete accounts, and escalate privileges

### Finding 4: Unsafe File Path Handling / Path Traversal
- **File:** `utils/files.py` (confirmed), `config.py` (false positive)
- **Scanner finding:** `os.path.join()` used with user-controlled input
- **Confirmed:** Partially -- confirmed in `utils/files.py`, dismissed in `config.py`

In [24]:
# Read utils/files.py to confirm the path traversal finding

print("utils/files.py -- download_file function:\n")
with open("utils/files.py", "r") as f:
    print(f.read())

# Read config.py to show why those flags are false positives
print("\nconfig.py -- flagged lines (false positive):\n")
with open("config.py", "r") as f:
    for i, line in enumerate(f.readlines(), start=1):
        if 16 <= i <= 26:
            print(f"{i:>4}  {line}", end="")

utils/files.py -- download_file function:

"""
SecureNote â€“ File Utilities
Helper functions for serving uploaded or generated files.
"""

import os
from flask import send_file, abort

from config import Config


def download_file(filename: str):
    """Serve a file from the uploads directory.

    Parameters
    ----------
    filename : str
        Name (or relative path) of the requested file.
    """
    filepath = os.path.join(Config.UPLOAD_FOLDER, filename)

    if not os.path.isfile(filepath):
        abort(404, description="File not found")

    return send_file(filepath, as_attachment=True)


config.py -- flagged lines (false positive):

  16  
  17      # â”€â”€ Database â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
  18      DATABASE_PATH = os.path.join(
  19          os.path.dirname(os.path.abspath(__file__)), "securenote.db"
  20      )
  

**Confirmed -- `utils/files.py`:**
- The `filename` parameter comes directly from the URL: `/files/<filename>`
- It is passed into `os.path.join()` with no sanitization or boundary check
- A request to `/files/../../config.py` resolves to the config file and serves it
- **Impact:** Arbitrary file read -- an attacker can read any file the server process has access to

**Dismissed -- `config.py` (false positive):**
- The scanner flagged `os.path.join()` calls on lines 18 and 23
- Both use `os.path.dirname(os.path.abspath(__file__))` as the base path
- This is a server-side constant -- no user input is involved at any point
- This is a false positive produced by the pattern-matching approach

### Finding 5: Hardcoded Secret Key
- **File:** `config.py`
- **Scanner finding:** `SECRET_KEY` assigned a hardcoded string value
- **Confirmed:** Yes

In [25]:
# Read config.py to confirm the hardcoded secret key

print("config.py -- SECRET_KEY:\n")
with open("config.py", "r") as f:
    for i, line in enumerate(f.readlines(), start=1):
        if 10 <= i <= 16:
            print(f"{i:>4}  {line}", end="")

config.py -- SECRET_KEY:

  10  class Config:
  11      """Base configuration."""
  12  
  13      # â”€â”€ Security â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
  14      SECRET_KEY = "supersecretkey123"                    # used for session signing
  15      SESSION_COOKIE_HTTPONLY = True
  16  


- Flask uses `SECRET_KEY` to sign session cookies
- If the key is known, an attacker can forge a valid session cookie for any user, including admin accounts
- The value `supersecretkey123` is short, guessable, and committed to the source code
- **Impact:** Session forgery -- an attacker who reads the source or config can impersonate any user

### Finding 6: Debug Mode Enabled in Production
- **File:** `config.py`
- **Scanner finding:** `DEBUG = True` present in the codebase
- **Confirmed:** Yes

In [26]:
# Read the ProductionConfig class to confirm debug mode is intentionally on

print("config.py -- ProductionConfig:\n")
with open("config.py", "r") as f:
    for i, line in enumerate(f.readlines(), start=1):
        if 32 <= i <= 44:
            print(f"{i:>4}  {line}", end="")

config.py -- ProductionConfig:

  32  
  33  class DevelopmentConfig(Config):
  34      """Development-specific settings."""
  35  
  36      DEBUG = True
  37  
  38  
  39  class ProductionConfig(Config):
  40      """Production settings â€“ used when FLASK_ENV=production."""
  41  
  42      DEBUG = True                                       # convenient for demos
  43      LOG_LEVEL = "INFO"
  44  


- The scanner flagged two `DEBUG = True` lines -- one in `DevelopmentConfig` and one in `ProductionConfig`
- The instance in `ProductionConfig` is the critical one; the inline comment reads "convenient for demos", confirming it is intentional
- Flask debug mode exposes the Werkzeug interactive debugger in the browser on any unhandled exception
- This allows arbitrary Python execution in the server process from the browser
- Full stack traces and local variable values are also exposed on every error
- **Impact:** Sensitive data exposure and potential RCE via the Werkzeug debugger in production

### Finding 7: Weak Password Hashing (MD5)
- **File:** `models/user.py`
- **Scanner finding:** `hashlib.md5()` used for password storage
- **Confirmed:** Yes

In [27]:
# Read models/user.py to confirm the MD5 hashing

print("models/user.py -- hash_password function:\n")
with open("models/user.py", "r") as f:
    for i, line in enumerate(f.readlines(), start=1):
        if 18 <= i <= 28:
            print(f"{i:>4}  {line}", end="")

models/user.py -- hash_password function:

  18  
  19  
  20  # â”€â”€ Password helpers â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
  21  
  22  def hash_password(password: str) -> str:
  23      """Return a hex-digest hash of *password* for storage."""
  24      return hashlib.md5(password.encode()).hexdigest()
  25  
  26  
  27  def verify_password(password: str, stored_hash: str) -> bool:
  28      """Check a plain-text *password* against a *stored_hash*."""


- Passwords are hashed with MD5 with no salt and no iterations
- MD5 was designed for speed, which is the wrong property for a password hash
- No salt means identical passwords produce identical hashes, enabling rainbow table attacks
- No iterations means there is no cost factor to slow down brute-force attempts
- **Impact:** Credential compromise -- if the database is leaked, passwords can be cracked quickly

## 4. Additional Findings from Manual Review

The following two vulnerabilities were not detected by the Phase 2 scanner. They were identified by reading the source code directly.

### Finding 8: Plaintext Password Logging
- **File:** `routes/auth.py`
- **Source:** Phase 1 manual review (not detected by scanner)
- **Confirmed:** Yes

In [28]:
# Read routes/auth.py to show the password logging line

print("routes/auth.py -- login function:\n")
with open("routes/auth.py", "r") as f:
    for i, line in enumerate(f.readlines(), start=1):
        if 60 <= i <= 76:
            print(f"{i:>4}  {line}", end="")

routes/auth.py -- login function:

  60  
  61      if get_user_by_username(username):
  62          return jsonify({"error": "Username already taken"}), 409
  63  
  64      user_id = create_user(username, password)
  65      logger.info("New user registered: id=%s, username=%s", user_id, username)
  66      return jsonify({"message": "User created", "user_id": user_id}), 201
  67  
  68  
  69  @auth_bp.route("/login", methods=["POST"])
  70  def login():
  71      """Authenticate and start a session."""
  72      data = request.get_json(force=True)
  73      username = data.get("username", "")
  74      password = data.get("password", "")
  75  
  76      logger.info(


- The login route logs the raw password on every attempt before verifying credentials
- This means failed login attempts are also logged, including cases where a user accidentally types their password in the username field
- The scanner rule for password logging did not match the exact format of this line
- **Impact:** Credential exposure -- anyone with access to the log file can read plaintext passwords

### Finding 9: Stored Cross-Site Scripting (XSS)
- **File:** `routes/notes.py`, `templates/note_detail.html`
- **Source:** Phase 1 manual review (not detected by scanner)
- **Confirmed:** Yes

In [29]:
# Show the route that renders the HTML template

print("routes/notes.py -- view_note function:\n")
with open("routes/notes.py", "r") as f:
    for i, line in enumerate(f.readlines(), start=1):
        if 68 <= i <= 78:
            print(f"{i:>4}  {line}", end="")

# Show the template line that disables escaping
print("\ntemplates/note_detail.html -- content rendering:\n")
with open("templates/note_detail.html", "r") as f:
    for i, line in enumerate(f.readlines(), start=1):
        if 18 <= i <= 24:
            print(f"{i:>4}  {line}", end="")

routes/notes.py -- view_note function:

  68  # â”€â”€ Read â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€â”€
  69  
  70  @notes_bp.route("/notes/<int:note_id>", methods=["GET"])
  71  @login_required
  72  def view_note(note_id):
  73      """View a single note. Renders HTML when Accept header includes text/html."""
  74      note = get_note(note_id)
  75      if note is None or note["user_id"] != session["user_id"]:
  76          return jsonify({"error": "Note not found"}), 404
  77  
  78      if "text/html" in request.headers.get("Accept", ""):

templates/note_detail.html -- content rendering:



UnicodeDecodeError: 'charmap' codec can't decode byte 0x90 in position 602: character maps to <undefined>

- The note detail route renders an HTML template when the request includes `Accept: text/html`
- The template renders `note.content` with the Jinja2 `| safe` filter, which explicitly disables HTML escaping
- Any JavaScript stored in a note's content field executes in the browser of any user who views it
- The scanner only scanned `.py` files and could not detect a vulnerability in an HTML template
- **Impact:** Stored XSS -- an attacker can store a script in a note that steals session cookies or credentials from other users who view it

## 5. Validation Table

A consolidated summary of all findings reviewed in Phase 3.

In [ ]:
import pandas as pd

validation_data = [
    {
        "#": 1,
        "Vulnerability": "Unsafe Deserialization",
        "File": "routes/export.py",
        "Confirmed": "Yes",
        "Impact": "Remote Code Execution"
    },
    {
        "#": 2,
        "Vulnerability": "Command Injection",
        "File": "routes/export.py",
        "Confirmed": "Yes",
        "Impact": "Remote Code Execution"
    },
    {
        "#": 3,
        "Vulnerability": "Missing Auth on Admin Routes",
        "File": "routes/admin.py",
        "Confirmed": "Yes",
        "Impact": "Unauthorized access, privilege escalation"
    },
    {
        "#": 4,
        "Vulnerability": "Path Traversal",
        "File": "utils/files.py",
        "Confirmed": "Yes",
        "Impact": "Arbitrary file read"
    },
    {
        "#": 5,
        "Vulnerability": "Hardcoded Secret Key",
        "File": "config.py",
        "Confirmed": "Yes",
        "Impact": "Session forgery"
    },
    {
        "#": 6,
        "Vulnerability": "Debug Mode in Production",
        "File": "config.py",
        "Confirmed": "Yes",
        "Impact": "Data exposure, potential RCE"
    },
    {
        "#": 7,
        "Vulnerability": "Weak Password Hashing (MD5)",
        "File": "models/user.py",
        "Confirmed": "Yes",
        "Impact": "Credential compromise"
    },
    {
        "#": 8,
        "Vulnerability": "Plaintext Password Logging",
        "File": "routes/auth.py",
        "Confirmed": "Yes",
        "Impact": "Credential exposure via logs"
    },
    {
        "#": 9,
        "Vulnerability": "Stored XSS",
        "File": "routes/notes.py, templates/note_detail.html",
        "Confirmed": "Yes",
        "Impact": "Script injection, session hijacking"
    },
    {
        "#": 10,
        "Vulnerability": "Path Traversal in config.py",
        "File": "config.py",
        "Confirmed": "No (false positive)",
        "Impact": "N/A"
    },
]

validation_table = pd.DataFrame(validation_data)
validation_table

### Findings

Phase 3 confirmed 9 real vulnerabilities across 6 files. One scanner finding was dismissed as a false positive.

The two most critical findings are unsafe deserialization and command injection in `routes/export.py`, both of which allow remote code execution by an authenticated user.

Two vulnerabilities -- plaintext password logging and stored XSS -- were not detected by the Phase 2 scanner. Password logging was identified in Phase 1 by reading the auth route. XSS was identified by reading the HTML template, which the scanner did not cover.

### Limitations

Phase 3 validation was limited to the files identified as high priority in Phase 1. Vulnerabilities in other files may exist but were not reviewed.

The analysis confirms a key limitation of the Phase 2 scanner: pattern matching can only detect what it was explicitly told to look for. It cannot follow data flow across functions or inspect template files. Manual review remains necessary to catch logic-level issues.

## 6. Findings Consolidation + Final CSV Export 

### Goal 
The goal of phase 4 is to transform the results from the previous phases into a seprate, clean, and structue vulnerability dataset.

By the end of this phases, we produce a finalized 'finding.csv' which:
1. consolidates findings from phase 2 - rule based analysis and phase 3 - LLM validation
2. removes false positives 
3. incorporates previously missed vulnerability 
4. Standardizes severity and confidence levels
5. Provides clear mitigation recommendations

Overall this file includes the authoritative output for all downstream analysis and reporting 

### Approach 

Our approach combines automation + validation + manual reasoning:

1. Start from validated findings (Phase 3)
2. Remove false positives identified during validation
3. Add vulnerabilities missed by automated tools
4. Merge all findings into a unified dataset
5. Remove duplicates across sources
6. Normalize severity and confidence values
7. Export a clean and structured findings.csv

This ensures both accuracy and completeness.

In [ ]:

# Convert PHase 2, Phase 3 table into Phase 4 schema
# Remove duplicate columns (KEEP FIRST occurrence)
phase2_df = pd.read_csv("findings.csv")   
phase3_df = validation_table.copy()
phase3_df = phase3_df[["File", "Impact", "Confirmed"]]

# Rename the columns
phase3_df = phase3_df.rename(columns={
    "File": "file",
    "Impact": "evidence",
    "Confirmed": "confirmed"
})

# Remove False Positive 
phase3_df = phase3_df[
    phase3_df["confirmed"] != "No (false positive)"
].copy()

# Normalize Paths
phase2_df["file"] = phase2_df["file"].str.replace("./", "", regex=False)
phase3_df["file"] = phase3_df["file"].astype(str)


# handle the multi file rows
phase3_df["file"] = phase3_df["file"].str.split(", ")
phase3_df = phase3_df.explode("file")

# merge phase 2 and phase 3
validated_df = pd.merge(
    phase3_df,
    phase2_df,
    on="file",
    how="left"
)

# Use Phase 3 explanation as primary evidence
validated_df["evidence"] = validated_df["evidence_x"]

# Drop duplicate columns
validated_df = validated_df.drop(columns=["evidence_x", "evidence_y"], errors="ignore")


validated_df["lines"] = validated_df["line_number"].fillna("N/A")
# validated_df["vulnerability_type"] = validated_df["vulnerability_type"].fillna("LLM-detected")

validated_df["mitigation"] = "See report"
# validated_df



In [ ]:
# # Remove the false positives & drop the unused column
# validated_df = validated_df[
#     validated_df["confirmed"] != "No (false positive)"
# ]
# # validated_df

In [ ]:
# Map severity from impact 
def map_severity(impact):
    impact = str(impact).lower()

    if "remote code execution" in impact:
        return "Critical"
    elif "unauthorized" in impact or "privilege" in impact:
        return "Critical"
    elif "session" in impact or "credential" in impact:
        return "High"
    elif "xss" in impact or "script injection" in impact:
        return "High"
    elif "file read" in impact:
        return "Medium"
    else:
        return "Low"

validated_df["severity"] = validated_df["evidence"].apply(map_severity)
# validated_df

In [ ]:
# Map confidence 
def map_confidence(evidence):
    evidence = str(evidence).lower()

    if "remote code execution" in evidence:
        return "Critical"
    elif "unauthorized" in evidence or "privilege" in evidence:
        return "High"
    elif "credential" in evidence:
        return "High"
    elif "session" in evidence:
        return "Med"
    elif "file read" in evidence:
        return "Med"
    elif "xss" in evidence or "script injection" in evidence:
        return "Med"
    else:
        return "Low"

validated_df["confidence"] = validated_df["evidence"].apply(map_confidence)
# validated_df

In [ ]:
# Manual Findings  
manual_findings = pd.DataFrame([
    {
        "file": "templates/note_detail.html",
        "lines": "template rendering",
        "severity": "High",
        "evidence": "Stored XSS vulnerability due to unsanitized HTML rendering",
        "confidence": "High",
        "mitigation": "Escape user input before rendering"
    },
    {
        "file": "routes/auth.py",
        "lines": "logging",
        "severity": "Medium",
        "evidence": "Plaintext password logging detected",
        "confidence": "High",
        "mitigation": "Do not log sensitive user credentials"
    }
])

In [ ]:
# Merge data
combined_df = pd.concat([validated_df, manual_findings], ignore_index=True)
# combined_df

In [ ]:
# Remove duplicates 

combined_df["key"] = combined_df["file"].astype(str) + "|" + combined_df["evidence"].astype(str)

combined_df = combined_df.drop_duplicates(subset="key").drop(columns="key")

combined_df

In [ ]:
# Normalize severity 
combined_df["severity"] = combined_df["severity"].str.capitalize()

def normalize_conf(val):
    val = str(val).lower()
    if "high" in val:
        return "High"
    elif "med" in val:
        return "Med"
    else:
        return "Low"

combined_df["confidence"] = combined_df["confidence"].apply(normalize_conf)
# combined_df

In [ ]:
# Normalize confidence 
def normalize_conf(val):
    val = str(val).lower()
    if "high" in val:
        return "High"
    elif "med" in val:
        return "Med"
    elif "low" in val:
        return "Low"
    else:
        return "Low"

combined_df["confidence"] = combined_df["confidence"].apply(normalize_conf)
# combined_df

In [ ]:
# Final Schema 
final_df = combined_df[[
    "file",
    "lines",
    "severity",
    "evidence",
    "confidence",
    "mitigation"
]]
# final_df

In [ ]:
# sort by severity 
severity_order = {"Critical": 0, "High": 1, "Medium": 2, "Low": 3}
final_df["rank"] = final_df["severity"].map(severity_order)
final_df = final_df.sort_values("rank").drop(columns="rank")
final_df

In [ ]:
# Export 
final_df.to_csv("findings.csv", index=False)

print(final_df)

## 7. Summary Statistics and Visualizations

These charts summarize the final validated findings and can be reused in the presentation deck.

In [ ]:
severity_counts = final_df["severity"].value_counts().reindex(["Critical", "High", "Medium", "Low"]).dropna()
severity_counts

In [ ]:
severity_counts.plot(kind="bar")
plt.title("Findings by Severity")
plt.xlabel("Severity")
plt.ylabel("Number of Findings")
plt.tight_layout()
plt.show()

In [ ]:
file_counts = final_df["file"].value_counts()
file_counts.plot(kind="bar")
plt.title("Findings by File")
plt.xlabel("File")
plt.ylabel("Number of Findings")
plt.tight_layout()
plt.show()

## 8. Verification and Limitations

**Verification performed:**

- The notebook scans the SecureNote repository, validates high-risk findings against source-code context, consolidates rows, removes duplicates, and exports `findings.csv`.
- The final CSV is constrained to the required schema: `file`, `lines`, `severity`, `evidence`, `confidence`, and `mitigation`.
- High-risk files such as `routes/export.py`, `routes/admin.py`, `utils/files.py`, `models/user.py`, and `routes/auth.py` were manually reviewed to confirm the most important findings.

**Limitations:**

- Regex-based static analysis can overflag patterns without full control-flow or data-flow context.
- Some vulnerabilities, especially missing authorization and stored XSS, require manual review of application behavior and route/template interactions.
- LLM- or manually-assisted validation improves explanation quality but does not replace security testing or exploit validation.
- Reproducibility can be affected by environment differences such as OS-specific path separators and file encoding defaults; this notebook normalizes paths and reads files with explicit encoding where needed.

## 9. Lessons Learned

- Simple rule-based scanning was effective for quickly surfacing high-risk issues in a small Flask codebase.
- LLM-assisted and manual validation were most useful for contextualizing findings and improving evidence quality, not replacing human review.
- Critical vulnerabilities often came from application logic and unsafe trust assumptions, such as missing admin authorization and unsafe import/export behavior.
- Consistent file handling, path normalization, and environment setup are essential for reproducible security analysis.
- Combining automated detection with manual validation produced a more credible final findings set than relying on either method alone.